# Kaggle Fine-tuning: mBART for Sinhala Spelling Correction

**Dataset:** SPEAK-ASR/akura-sinhala-dyslexia-corrected

**Model:** facebook/mbart-large-50

**Training Time:** ~3-4 hours

This notebook fine-tunes mBART model for Sinhala spelling correction using real dyslexia correction data. mBART provides better quality but takes longer to train.

## Step 1: Install Dependencies

In [1]:
!pip install -q datasets huggingface-hub wandb transformers sentencepiece accelerate peft numpy
!pip install -q torch==2.9.0+cu126 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
print("✓ Dependencies installed successfully")


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
✓ Dependencies installed successfully


## Step 2: Import Libraries and Setup

In [2]:
import os
import torch
import logging
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
hf_token = os.environ.get("HF_TOKEN", "").strip()
if hf_token:
    login(token=hf_token)
    print("✓ Hugging Face logged in via HF_TOKEN")
else:
    print("⚠️ No HF_TOKEN found. Set it in environment or .env")

In [ ]:
import os
from dotenv import load_dotenv
import wandb

load_dotenv()
wandb_key = os.environ.get("WANDB_API_KEY", "").strip()
if wandb_key:
    wandb.login(key=wandb_key)

run = wandb.init(
    entity="SPEAK-ASR-uom",
    project="finetune-mbart-large-si",
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: iruda-21 (SPEAK-ASR-uom) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Step 3: Configuration

In [5]:
# Kaggle paths
KAGGLE_INPUT = "/content/input"
KAGGLE_OUTPUT = "/content/working"

# Configuration
CONFIG = {
    "model_name": "facebook/mbart-large-50",
    "dataset_id": "SPEAK-ASR/akura-sinhala-dyslexia-corrected",
    "source_lang": "si_LK",
    "target_lang": "si_LK",
    "max_input_length": 128,
    "max_target_length": 128,
    "batch_size": 32,
    "num_epochs": 5,
    "learning_rate": 5e-5,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "data_train_percentage": 0.7,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"\n{'='*70}")
print("mBART SINHALA SPELLING CORRECTION - KAGGLE FINE-TUNING")
print(f"{'='*70}\n")
print(f"Device: {device}")
print(f"Dataset: {CONFIG['dataset_id']}")
print(f"Model: {CONFIG['model_name']}")
print(f"Language: {CONFIG['source_lang']}")
print(f"Output: {KAGGLE_OUTPUT}")
print(f"Data Split: {CONFIG['data_train_percentage']*100:.0f}% train, {(1-CONFIG['data_train_percentage'])*100:.0f}% test\n")


mBART SINHALA SPELLING CORRECTION - KAGGLE FINE-TUNING

Device: cuda
Dataset: SPEAK-ASR/akura-sinhala-dyslexia-corrected
Model: facebook/mbart-large-50
Language: si_LK
Output: /content/working
Data Split: 70% train, 30% test



## Step 4: Load Dataset

In [6]:
print(f"[STEP 1/5] Loading Dataset from Hugging Face Hub...")
try:
    dataset = load_dataset(CONFIG["dataset_id"])
    print(f"✓ Dataset loaded successfully")
    print(f"  Train samples: {len(dataset['train'])}")
    if 'test' in dataset:
        print(f"  Test samples: {len(dataset['test'])}")

    # Display sample
    print(f"\nSample data:")
    sample = dataset['train'][0]
    for key, value in sample.items():
        print(f"  {key}: {value}")
except Exception as e:
    print(f"✗ Error loading dataset: {e}")
    raise

[STEP 1/5] Loading Dataset from Hugging Face Hub...
✓ Dataset loaded successfully
  Train samples: 23030

Sample data:
  clean_sentence: ගානට සල්ලි
  dyslexic_sentence: ගාණට සල්ලි
  error_type: Phonetic Confusion


In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['clean_sentence', 'dyslexic_sentence', 'error_type'],
        num_rows: 23030
    })
})

## Step 5: Load Model and Tokenizer

In [8]:
print(f"\n[STEP 2/5] Loading Model and Tokenizer...")
try:
    tokenizer = AutoTokenizer.from_pretrained(
        CONFIG["model_name"],
        use_fast=False
    )
    # Set languages AFTER loading
    tokenizer.src_lang = CONFIG["source_lang"]
    tokenizer.tgt_lang = CONFIG["target_lang"]
    model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"])

    # Set language tokens for mBART - Convert target language token to ID
    lang_token_id = tokenizer.convert_tokens_to_ids([CONFIG["target_lang"]])[0]
    model.config.decoder_start_token_id = lang_token_id
    model.config.forced_bos_token_id = lang_token_id

    print(f"✓ Model loaded")
    print(f"  Vocab size: {len(tokenizer)}")
    print(f"  Source language: {CONFIG['source_lang']}")
    print(f"  Target language: {CONFIG['target_lang']}")

    model = model.to(device)
except Exception as e:
    print(f"✗ Error loading model: {e}")
    raise


[STEP 2/5] Loading Model and Tokenizer...
✓ Model loaded
  Vocab size: 250054
  Source language: si_LK
  Target language: si_LK


## Step 6: Preprocess and Tokenize Data

In [9]:
print(f"\n[STEP 3/6] Tokenizing Datasets...")

# First, inspect the dataset structure
print("\n[DEBUG] Inspecting dataset structure:")
print(f"  Dataset keys: {dataset.keys()}")
print(f"  Train columns: {dataset['train'].column_names}")
print(f"  First example: {dataset['train'][0]}")
print(f"  Train size: {len(dataset['train'])}")

def preprocess_function(examples):
    # Handle different column names
    input_texts = []
    target_texts = []
    
    # Determine column names
    input_col = "input_text" if "input_text" in examples else "dyslexic_sentence"
    target_col = "corrected_text" if "corrected_text" in examples else "clean_sentence"
    
    # Extract texts
    for i in range(len(examples[input_col])):
        input_text = examples[input_col][i]
        target_text = examples[target_col][i]
        
        # Filter out None or empty strings
        if input_text and target_text and len(input_text.strip()) > 0 and len(target_text.strip()) > 0:
            input_texts.append(input_text)
            target_texts.append(target_text)
    
    # If no valid examples, return empty dict with correct structure
    if not input_texts:
        return {
            "input_ids": [],
            "attention_mask": [],
            "labels": []
        }
    
    # Tokenize inputs
    model_inputs = tokenizer(
        input_texts,
        max_length=CONFIG["max_input_length"],
        padding="max_length",
        truncation=True,
    )
    
    # Tokenize targets with target tokenizer
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            target_texts,
            max_length=CONFIG["max_target_length"],
            padding="max_length",
            truncation=True,
        )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

try:
    # Process dataset
    print("\n[DEBUG] Starting tokenization...")
    dataset_processed = dataset.map(
        preprocess_function,
        batched=True,
        batch_size=100,
        desc="Tokenizing",
        remove_columns=dataset['train'].column_names  # Remove columns during mapping
    )
    
    # Check if dataset is empty
    print(f"\n[DEBUG] After tokenization:")
    print(f"  Train size: {len(dataset_processed['train'])}")
    
    if len(dataset_processed['train']) == 0:
        raise ValueError("Dataset is empty after tokenization! Check your input data.")
    
    # Filter out any empty examples (belt and suspenders approach)
    def filter_empty(example):
        return (
            example['input_ids'] is not None 
            and len(example['input_ids']) > 0
            and example['labels'] is not None
            and len(example['labels']) > 0
        )
    
    dataset_processed = dataset_processed.filter(filter_empty)
    
    print(f"\n[DEBUG] After filtering:")
    print(f"  Train size: {len(dataset_processed['train'])}")
    print(f"  Columns: {dataset_processed['train'].column_names}")
    print(f"  Sample: {dataset_processed['train'][0]}")
    
    # -------------------------------
    # SPLIT: Train 70%, Eval 10%, Test 20%
    # -------------------------------
    # First split: 70% train, 30% temp
    train_temp_split = dataset_processed["train"].train_test_split(
        test_size=0.30,
        seed=42
    )
    train_dataset = train_temp_split["train"]
    temp_dataset = train_temp_split["test"]
    
    # Second split: temp -> 10% eval, 20% test
    # (eval = 1/3 of temp, test = 2/3 of temp)
    eval_test_split = temp_dataset.train_test_split(
        test_size=2/3,
        seed=42
    )
    eval_dataset = eval_test_split["train"]
    test_dataset = eval_test_split["test"]
    
    print("\n✓ Datasets tokenized and split")
    print(f"  Train: {len(train_dataset)} samples (70%)")
    print(f"  Eval:  {len(eval_dataset)} samples (10%)")
    print(f"  Test:  {len(test_dataset)} samples (20%)")
    print(f"  Total: {len(train_dataset) + len(eval_dataset) + len(test_dataset)}")
    
    # Final verification
    print(f"\n[DEBUG] Final dataset check:")
    print(f"  Train columns: {train_dataset.column_names}")
    print(f"  Expected columns: ['input_ids', 'attention_mask', 'labels']")
    
except Exception as e:
    print(f"✗ Error tokenizing datasets: {e}")
    import traceback
    traceback.print_exc()
    raise


[STEP 3/6] Tokenizing Datasets...

[DEBUG] Inspecting dataset structure:
  Dataset keys: dict_keys(['train'])
  Train columns: ['clean_sentence', 'dyslexic_sentence', 'error_type']
  First example: {'clean_sentence': 'ගානට සල්ලි', 'dyslexic_sentence': 'ගාණට සල්ලි', 'error_type': 'Phonetic Confusion'}
  Train size: 23030

[DEBUG] Starting tokenization...


Tokenizing:   0%|          | 0/23030 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(



[DEBUG] After tokenization:
  Train size: 23008


Filter:   0%|          | 0/23008 [00:00<?, ? examples/s]


[DEBUG] After filtering:
  Train size: 23008
  Columns: ['input_ids', 'attention_mask', 'labels']
  Sample: {'input_ids': [250022, 50798, 7780, 722, 89758, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [250022, 124242, 722, 89758, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

## Step 7: Setup Training

In [10]:
print(f"\n[STEP 4/5] Setting up Training...")

output_dir = os.path.join(KAGGLE_OUTPUT, "mbart-model")
os.makedirs(output_dir, exist_ok=True)

eval_steps = 500

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    # save_total_limit=2,
    logging_steps=1,
    generation_max_length=CONFIG["max_target_length"],
    fp16=False,
    bf16=True,
    predict_with_generate=True,
    report_to=["wandb"],
    dataloader_num_workers=8,
    dataloader_pin_memory=True,
    push_to_hub=True,
    hub_strategy="checkpoint",
    hub_model_id="SPEAK-ASR/mBART-large-50-si-2",
    # neftune_noise_alpha=5.0,
    remove_unused_columns=True,
    label_names=["labels"],
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


[STEP 4/5] Setting up Training...


## Step 8: Start Training

In [11]:
# from peft import get_peft_model, LoraConfig
# from peft import prepare_model_for_kbit_training

# # Prepare the model for LoRA-compatible 8-bit training (freezing norms, casting types)
# model = prepare_model_for_kbit_training(model)

# # Configure LoRA (Low-Rank Adaptation) for efficient fine-tuning
# config = LoraConfig(
#     r=32,  # Rank of LoRA decomposition
#     lora_alpha=64,  # Scaling factor
#     target_modules=["q_proj", "v_proj"],  # Apply LoRA to attention projections
#     lora_dropout=0.05,  # Dropout applied to LoRA layers
#     bias="none"  # Don't adapt bias terms
# )

# # Wrap the base model with LoRA using the above config
# model = get_peft_model(model, config)
# model.print_trainable_parameters()  # Print which parameters are trainable

In [12]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"✓ Training setup complete")

✓ Training setup complete


In [16]:
print(f"\n[STEP 5/5] Starting Training...\n")

try:
    train_result = trainer.train()
    print(f"\n✓ TRAINING COMPLETED SUCCESSFULLY!")
    print(f"  Total training loss: {train_result.training_loss:.4f}")
    print(f"  Model saved to: {output_dir}")
except KeyboardInterrupt:
    print(f"\n⚠️  Training interrupted by user")
except Exception as e:
    print(f"\n✗ Error during training: {e}")
    raise


[STEP 5/5] Starting Training...



There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



✗ Error during training: 'NoneType' object has no attribute 'load_state_dict'


AttributeError: 'NoneType' object has no attribute 'load_state_dict'

## Step 10: Evaluate on Test Set

In [ ]:
import numpy as np

print(f"\n[STEP 6/6] Evaluating Model on Test Set...\n")

try:
    # Generate predictions on the test set
    predictions = trainer.predict(test_dataset)

    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions.predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(predictions.label_ids, skip_special_tokens=True)

    # Display sample predictions
    print("📊 Sample Predictions (First 5 test examples):")
    print("=" * 100)
    for i in range(min(5, len(decoded_preds))):
        print(f"\nExample {i+1}:")
        print(f"  Input (Dyslexic):  {decoded_preds[i]}")
        print(f"  Expected (Clean):  {decoded_labels[i]}")
        print(f"  Match: {'✓' if decoded_preds[i].strip() == decoded_labels[i].strip() else '✗'}")

    # Calculate character-level accuracy
    char_accuracies = []
    for pred, label in zip(decoded_preds, decoded_labels):
        matches = sum(1 for p, l in zip(pred, label) if p == l)
        total = max(len(pred), len(label), 1)
        char_accuracies.append(matches / total)

    avg_char_accuracy = np.mean(char_accuracies)

    # Calculate exact match accuracy
    exact_matches = sum(1 for pred, label in zip(decoded_preds, decoded_labels)
                        if pred.strip() == label.strip())
    exact_match_accuracy = exact_matches / len(decoded_preds)

    # Print evaluation results
    print("\n" + "=" * 100)
    print("📈 EVALUATION RESULTS:")
    print("=" * 100)
    print(f"Data Split: {CONFIG['data_train_percentage']*100:.0f}% Train, {(1-CONFIG['data_train_percentage'])*100:.0f}% Test")
    print(f"Test Set Size: {len(eval_dataset)} samples")
    print(f"\nMetrics:")
    print(f"  • Character-Level Accuracy: {avg_char_accuracy*100:.2f}%")
    print(f"  • Exact Match Accuracy: {exact_match_accuracy*100:.2f}% ({exact_matches}/{len(decoded_preds)})")
    print("=" * 100)

    print("\n✓ Model evaluation completed successfully!")
    print(f"\nNote: The model was trained on {len(train_dataset)} samples and evaluated on {len(eval_dataset)} samples")

except Exception as e:
    print(f"✗ Error during evaluation: {e}")
    raise

## Step 9: Save Model

In [ ]:
kwargs = {
    "dataset_tags": CONFIG["dataset_id"],
    "dataset": CONFIG["dataset_id"],  # a 'pretty' name for the training dataset
    "language": "si",
    "model_name": "SPEAK-ASR/mBART-large-50-si-2",  # a 'pretty' name for your model
    "finetuned_from": CONFIG["model_name"],
}

trainer.push_to_hub(**kwargs)

In [ ]:
print(f"\n[SAVING] Saving final model...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✓ Model and tokenizer saved to {output_dir}")

print(f"\n{'='*70}")
print("✓ TRAINING PIPELINE COMPLETE!")
print(f"{'='*70}")
print(f"\nNext steps:")
print(f"1. Download model from: {output_dir}")
print(f"2. Load with: AutoTokenizer.from_pretrained('{output_dir}')")
print(f"3. Use for inference on Sinhala spelling correction")